# PropertyScan AI — Colab T4 Pipeline

End-to-end run on **Google Colab Free (Tesla T4)** after all phases.

**Modes**
1. **Mock** (no heavy deps) — verify plumbing
2. **Real** — MASt3R + Depth Anything + splatfacto (install cells below)

Upload a walkthrough video or a folder of frames, then run cells in order.

## 1) Runtime check

In [ ]:
!nvidia-smi -L || true
import sys
print(sys.version)

## 2) Get the code

Option A: mount Drive and `cd` to your clone of `propertyscan-ai/revamped_code`.

Option B: upload a zip of `revamped_code` and extract it.

In [ ]:
# Example: if project is on Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/propertyscan-ai/revamped_code

import os
from pathlib import Path
ROOT = Path.cwd()
print('ROOT', ROOT)
assert (ROOT / 'propertyscan').is_dir() or (ROOT / 'revamped_code' / 'propertyscan').is_dir()
if (ROOT / 'revamped_code' / 'propertyscan').is_dir():
    %cd revamped_code
    ROOT = Path.cwd()
print('Using', ROOT)

## 3) Install PropertyScan (always)

In [ ]:
!pip -q install -e ".[dev]"
!propertyscan version
!propertyscan doctor --profile colab_t4

## 4) Optional: install real geometry stack (Phase 4)

Skip this section for mock-only smoke tests.

Install PyTorch CUDA, transformers, then NAVER dust3r + mast3r (see docs/PHASE4_WALKTHROUGH.md).

In [ ]:
# Uncomment for REAL runs on T4
# !pip -q install torch torchvision --index-url https://download.pytorch.org/whl/cu121
# !pip -q install transformers einops
# !git clone --recursive https://github.com/naver/dust3r /content/dust3r
# !pip -q install -e /content/dust3r
# !git clone --recursive https://github.com/naver/mast3r /content/mast3r
# !pip -q install -e /content/mast3r
# !propertyscan doctor --profile colab_t4

## 5) Provide input media

In [ ]:
from pathlib import Path

# Point this at your video or frames folder after upload
INPUT = Path('tests/fixtures/frames')  # replace with /content/walk.mp4 etc.
OUT = Path('_colab_out')
OUT.mkdir(exist_ok=True)
assert INPUT.exists(), INPUT
print(INPUT, '->', OUT)

## 6a) Mock full pipeline (no GPU models)

In [ ]:
!propertyscan export --input {INPUT} --out {OUT}/mock_run --engine mock --train-backend mock --profile colab_t4
print('Done. See', OUT / 'mock_run')

## 6b) Real MASt3R path (after section 4)

Uses `colab_t4` profile: swin pair graph, depth small, capped train iters when dense init exists.
Requires `ns-train` for real splatfacto (optional install Nerfstudio).

In [ ]:
# Uncomment for REAL geometry (needs section 4)
# !propertyscan geometry --input {INPUT} --out {OUT}/real_geo --engine mast3r --profile colab_t4
# Full export with real train backend when nerfstudio is installed:
# !propertyscan export --input {INPUT} --out {OUT}/real_full --engine mast3r --train-backend splatfacto --profile colab_t4

## 7) Benchmark + research layout

In [ ]:
# Place multiple scene folders under data/bench_scenes/<scene_id>/
# !propertyscan benchmark --data tests/fixtures --out {OUT}/bench --engine mock --train-backend mock
print('Use benchmark when you have multi-scene folders ready.')

## Done

Artifacts to download: `scene.ply`, `property_scene.json`, `final_report.json`, `provenance.json`.